In [0]:
%uv pip install shapely

In [0]:
import json

import pandas as pd
from pyspark.sql.functions import col, pandas_udf, regexp_replace, upper, when
from pyspark.sql.types import StringType
from shapely.geometry import Point, shape

In [0]:
postalcode_regex = r"^[0-9]{4}[A-Z]{2}$"
postalcode_regex_with_space = r"^[0-9]{4} [A-Z]{2}$"

In [0]:
# Read the airbnb.csv dataset and display it
df_airbnb = spark.read.csv(
    "/Workspace/Users/fl.casciaro@gmail.com/revodatanl-assessment/data/airbnb.csv",
    header=True,
    inferSchema=True,
).withColumn("zipcode", upper("zipcode"))
display(df_airbnb)

In [0]:
df_airbnb.count()

In [0]:
airbnb_invalid_count = df_airbnb.filter(
    col("zipcode").isNull() | ~col("zipcode").rlike(postalcode_regex)
).count()

print(f"Not valid: {airbnb_invalid_count}")

In [0]:
airbnb_cleaned = df_airbnb.filter(
    col("zipcode").rlike(postalcode_regex) | col("zipcode").rlike(postalcode_regex_with_space)
).withColumn("zipcode", regexp_replace(col("zipcode"), "^([0-9]{4}) ([a-zA-Z]{2})$", "$1$2"))

In [0]:
display(airbnb_cleaned)

In [ ]:
from pathlib import Path

with Path(
    "/Workspace/Users/fl.casciaro@gmail.com/revodatanl-assessment/data/geo/post_codes.geojson"
).open() as f:
    geojson_data = json.load(f)

polygons = []
for feature in geojson_data["features"]:
    geom = shape(feature["geometry"])
    pc4_code = feature["properties"]["pc4_code"]
    polygons.append((geom, pc4_code))

print(f"Loaded {len(polygons)} postal code polygons")
print(f"Sample: pc4_code={polygons[0][1]}, area={polygons[0][0].area:.6f}")

In [ ]:
@pandas_udf(StringType())
def lookup_postcode(longitudes: pd.Series, latitudes: pd.Series) -> pd.Series:
    """Look up the 4-digit postal code (pc4_code) for each lat/lon point."""
    results = []
    for lon, lat in zip(longitudes, latitudes, strict=True):
        if pd.isna(lon) or pd.isna(lat):
            results.append(None)
            continue
        point = Point(float(lon), float(lat))
        found = None
        for geom, pc4 in polygons:
            if geom.contains(point):
                found = pc4
                break
        results.append(found)
    return pd.Series(results)


# Enrich rows with null zipcode using geospatial lookup
# Note: pc4_code is the 4-digit prefix only (e.g. "1052"), not the full Dutch code "1052AB"
airbnb_enriched = df_airbnb.withColumn(
    "zipcode",
    when(col("zipcode").isNull(), lookup_postcode(col("longitude"), col("latitude"))).otherwise(
        col("zipcode")
    ),
)

display(airbnb_enriched.filter(col("zipcode").isNotNull()))

In [0]:
# Read the rentals.json dataset and display it
df_rentals = spark.read.json(
    "/Workspace/Users/fl.casciaro@gmail.com/revodatanl-assessment/data/rentals.json"
).withColumn("postalCode", upper("postalCode"))
display(df_rentals)

In [0]:
df_rentals.count()

In [0]:
rentals_invalid_count = df_rentals.filter(
    col("postalCode").isNull() | ~col("postalCode").rlike(postalcode_regex)
).count()

print(f"Not valid: {rentals_invalid_count}")